# Task 2: Predicting severity metrics
Run this notebook to predict the severity (UPDRSM). 

Structure of this notebook: 
1. Set up
2. Model 1: Multiple Linear Regression (from `src/models/linreg.py`) and Performance 
3. Model 2: Random Forest (from `src/models/rf.py`) and Performance
4. Comparing Model 1 and Model 2
5. Model 3: 1DCNN (from `src/models/cnn1d.py`) and Performance
6. Comparing across all 3 models 

## 1. Set up and data preparation

In [ ]:
# RUN CODE: execute this cell first, once per new kernel.
from pathlib import Path  
import sys  
import numpy as np  
import pandas as pd 
import joblib  # Save fitted classical models for reproducibility.
import matplotlib.pyplot as plt

# CHECK PATH: start Jupyter in repo root OR in repo/notebooks.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').exists(), f'Cannot locate repo src/ folder from {Path.cwd()}' # checks that models exist in src
sys.path.insert(0, str(ROOT))

FEATURES_PATH = ROOT / 'data' / 'features.csv' # to check if correct once features.csv is in 
METADATA_PATH = ROOT / 'data' / 'raw' / 'others' / 'subject_metadata.xls' # to check once processed
SPLITS_PATH = ROOT / 'data' / 'splits.json'

# saving output in folder
OUT = ROOT / 'results' / 'task2' 
(OUT / 'predictions').mkdir(parents=True, exist_ok=True)
(OUT / 'weights').mkdir(parents=True, exist_ok=True)
(OUT / 'figures').mkdir(parents=True, exist_ok=True)

In [ ]:
# check metadata and print 
metadata = pd.read_excel(METADATA_PATH)  
print('Metadata dimensions:', metadata.shape)
print('Metadata columns:', metadata.columns.tolist())
display(metadata.head(5))  

assert {'Study', 'Group', 'Subjnum', 'UPDRSM'}.issubset(metadata.columns) # check for required rows

# check features and print
features = pd.read_csv(FEATURES_PATH)
print('Feature table dimensions:', features.shape)
print('Feature columns:', features.columns.tolist())
display(features.head(5))

### Merge metadata with features
The metadata contains our target UPDRSM, whereas features contain the gait characteristics used for prediction. We merge the dataframes. 

NOTE: to check if need to conduct checks on the metadata and features here, might not need to but already added in case. WORK IN PROGRESS!! will change based on finalised features.csv

In [ ]:
metadata = metadata.copy()

# check meatadata data frame (unsure if this is needed)
if 'ID' in metadata.columns:
    metadata['subject'] = metadata['ID'].astype(str).str.strip()
else:
    metadata['subject'] = (metadata['Study'].astype(str).str.strip() + metadata['Group'].astype(str).str.strip()
                           + pd.to_numeric(metadata['Subjnum'], errors='raise').astype(int).astype(str).str.zfill(2))
assert metadata['subject'].is_unique, 'Metadata has duplicate subject IDs.'

# check features data frame (unsure if this is needed) 
if 'subject' not in features.columns:
    file_options = [c for c in ('file','filename','walk_file') if c in features.columns]
    assert file_options, 'CHECK COLUMN: features.csv must have subject or filename.'
    features['subject'] = features[file_options[0]].astype(str).map(lambda x: Path(x).stem[:6])
features['subject'] = features['subject'].astype(str).str.strip()
assert features['subject'].str.match(r'^(Ga|Ju|Si)(Co|Pt)\d{2}$').all(), 'Unexpected subject format.'

# merge metadata and features
merged = features.merge(metadata, on='subject', how='left', validate='many_to_one', indicator=True,
                        suffixes=('', '_metadata'))
print('Merge matches:'); display(merged['_merge'].value_counts().to_frame('rows'))
assert (merged['_merge'] == 'both').all(), 'Unmatched feature rows; inspect subject IDs.'

# keep patients with a valid motor-severity target only 
merged['UPDRSM'] = pd.to_numeric(merged['UPDRSM'], errors='coerce')
patients = merged.loc[merged['subject'].str[2:4].eq('Pt') & merged['UPDRSM'].notna()].copy().reset_index(drop=True)
assert len(patients)>0
assert patients.groupby('subject')['UPDRSM'].nunique().max()==1

# checkpoints
print('Eligible patient rows:',len(patients),'unique subjects:',patients.subject.nunique())
display(patients[['subject','UPDRSM']].head(10))

for below code: need to check features that they match

In [ ]:
# select only numerical gait features.
# exclude metadata, labels, IDs, speed and other clinical scores to avoid target leakage.
metadata_columns = set(metadata.columns) | {'subject', '_merge', 'file', 'filename', 'walk_file',
    'walk', 'study', 'group', 'label', 'target', 'UPDRS', 'UPDRSM', 'HoehnYahr', 'TUAG',
    'Speed_01 (m/sec)', 'Speed_10 (m/sec)', 'Speed_01', 'Speed_10'}

# check for numerical feature columns not in the exclusion set 
feature_columns = [c for c in features.columns if c not in metadata_columns
                   and c in patients.columns and pd.api.types.is_numeric_dtype(patients[c])]
print('Selected features:', feature_columns)
assert feature_columns, 'No numeric feature columns found; inspect features.csv column names.'
assert 'UPDRSM' not in feature_columns, 'Target leakage!' # make sure predictor not in features 

# create feature matrix 
X = patients[feature_columns].replace([np.inf, -np.inf], np.nan) 

# target vector 
y = patients['UPDRSM'].astype(float)

# checkpoints
assert y.notna().all() and np.isfinite(y).all(), 'Invalid target values.'
print('Feature matrix shape:', X.shape, '| Target shape:', y.shape)
print('Missing values by feature:')
display(X.isna().sum().sort_values(ascending=False).head(10).to_frame('missing'))
display(X.head(5))

Setting up model training and model performance helper functions to use for each model. 


!!work in progress!!

In [ ]:
import json
from collections import Counter

from src.evaluate import regression_metrics  # to create evaluate file to calculate metrics later on!!! 
from src.models.linreg import make_linear_regression  # MLR 
from src.models.rf import make_random_forest  # rf
from sklearn.dummy import DummyRegressor  # for mean baseline

# load splits
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task2"]
test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_idx = np.flatnonzero(patients["subject"].isin(fold["train"]).to_numpy())
    val_idx = np.flatnonzero(patients["subject"].isin(fold["val"]).to_numpy())
    
    # ensure proper split 
    assert not set(patients.iloc[train_idx]["subject"]) & test_subjects
    assert not set(patients.iloc[val_idx]["subject"]) & test_subjects
    assert not set(patients.iloc[train_idx]["subject"]) & \
               set(patients.iloc[val_idx]["subject"])

    folds.append((train_idx, val_idx))
    
assert len(folds) == 5 # check num folds 

val_subject_counts = Counter(
    subject
    for fold in assignment["folds"]
    for subject in fold["val"]
)

development_subjects = set(patients["subject"]) - test_subjects

assert set(val_subject_counts) == development_subjects
assert all(count == 1 for count in val_subject_counts.values())

print("Cross-validation setup complete.")
print("Number of folds:", len(folds))
print("Number of held-out test subjects:", len(test_subjects))

results_by_model={}; metrics_by_model={}
def train_classical_model(model_name, model_factory):
    # Train on each saved train split, evaluate on that fold's VALIDATION split.
    tables, rows=[],[]
    for k,(tr,va) in enumerate(folds,1):
        model=model_factory()
        model.fit(X.iloc[tr],y.iloc[tr])
        pred=np.asarray(model.predict(X.iloc[va])).reshape(-1)
        assert len(pred)==len(va) and np.isfinite(pred).all()
        scores=regression_metrics(y.iloc[va],pred)
        table=pd.DataFrame({'model':model_name,'fold':k,'subject':patients.iloc[va]['subject'].to_numpy(),
                            'actual_UPDRSM':y.iloc[va].to_numpy(),'predicted_UPDRSM':pred})
        rows.append({'model':model_name,'fold':k,**scores}); tables.append(table)
        dest=OUT/'predictions'/f'{model_name}_val_fold{k}.csv'
        table.to_csv(dest,index=False)
        joblib.dump(model,OUT/'weights'/f'{model_name}_val_fold{k}.joblib')
        print(f'{model_name} validation fold {k}: MAE={scores["MAE"]:.3f}, RMSE={scores["RMSE"]:.3f}, saved={dest.exists()}')
        display(table.head(3))
    results_by_model[model_name]=pd.concat(tables,ignore_index=True)
    metrics_by_model[model_name]=pd.DataFrame(rows)
    return results_by_model[model_name],metrics_by_model[model_name]

def show_regression_performance(model_name):
    table=results_by_model[model_name]
    # Only CV subjects are expected here; the held-out test remains untouched.
    assert set(table.subject)==cv_subjects
    display(metrics_by_model[model_name])
    scores=regression_metrics(table.actual_UPDRSM,table.predicted_UPDRSM)
    print(model_name,'pooled CV validation scores:',scores)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    a,p=table.actual_UPDRSM,table.predicted_UPDRSM
    axes[0].scatter(a,p,alpha=.6)
    low=min(a.min(),p.min()); high=max(a.max(),p.max())
    axes[0].plot([low,high],[low,high],'--')
    axes[0].set(xlabel='Actual UPDRSM',ylabel='Predicted UPDRSM',title='CV validation predictions')
    axes[1].scatter(p,a-p,alpha=.6)
    axes[1].axhline(0,linestyle='--')
    axes[1].set(xlabel='Predicted UPDRSM',ylabel='Residual',title='CV residuals')
    fig.tight_layout(); fig.savefig(OUT/'figures'/f'{model_name}_cv_diagnostics.png',dpi=160)
    plt.show()
    return scores

### Mean baseline 
This is used as a reference point. Ideally, our models should perform better than this.

In [ ]:
baseline_predictions, baseline_metrics = train_classical_model(
    'mean_baseline', lambda: DummyRegressor(strategy='mean'))
print('Baseline model check:')
display(baseline_metrics)
baseline_scores = show_regression_performance('mean_baseline')

## 2. Model 1: Multiple Linear Regression

In [ ]:
# CHECK COLUMNS: Feature extraction must supply at least two numeric gait predictors.
print('Number of predictors:', X.shape[1])  # One column = one input feature.
print('Number of walking trials:', X.shape[0])  # One row = one observation.
print('Feature names:', list(X.columns))  # Inspect for IDs, UPDRSM or other leakage.
display(X.head())  # Preview the actual values fed to the regression.
assert X.shape[1] >= 2, 'MLR requires at least two predictor columns.'
assert 'UPDRSM' not in X.columns, 'Do not use UPDRSM as an input feature.'
print('MLR input check passed:', X.shape[1] >= 2 and 'UPDRSM' not in X.columns)

In [ ]:
mlr_predictions, mlr_metrics = train_classical_model('linear_regression', make_linear_regression)
print('MLR trained folds:', mlr_metrics['fold'].tolist())

Performance of MLR:

In [ ]:
mlr_scores = show_regression_performance('linear_regression')
# Load one fold's fitted pipeline to inspect its regression coefficients.
mlr_fold1 = joblib.load(OUT / 'weights' / 'linear_regression__val_fold1.joblib')
coefs = pd.DataFrame({'feature': feature_columns,
                      'standardised_coefficient': mlr_fold1.named_steps['model'].coef_})
coefs['absolute_coefficient'] = coefs['standardised_coefficient'].abs()

# outputs 
print('MLR intercept (fold 1):', mlr_fold1.named_steps['model'].intercept_)
display(coefs.sort_values('absolute_coefficient', ascending=False).head(15))

# checkpoints
assert len(coefs) == X.shape[1], 'Coefficient count does not match feature count.' 

## Model 2: Random Forest